In [4]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, GlobalAveragePooling1D, LSTM
from tensorflow.keras.models import Sequential

## Étape 1 — Vérification de l’environnement et des données

Avant de commencer l’analyse, nous vérifions la version de Python, la disponibilité des bibliothèques nécessaires et la présence du fichier de données avec les colonnes attendues.

**Objectif :** repérer dès le départ les problèmes d’environnement ou de format qui pourraient interrompre les étapes suivantes.

In [5]:
print(f"Version de Python : {sys.version.split()[0]}")
if sys.version_info > (3, 13, 5):
    print("Attention : les consignes recommandent Python 3.13.5 ou une version antérieure pour Keras/TensorFlow.")

packages = ["pandas", "numpy", "scikit-learn", "nltk", "tensorflow", "torch", "gensim"]
for package in packages:
    try:
        print(f"{package} : {version(package)}")
    except PackageNotFoundError:
        print(f"{package} : non installé")

DATA_PATH = Path("bbc-news-data.csv")
if not DATA_PATH.is_file():
    DATA_PATH = Path("Apply SVM to predict BBC news category") / "bbc-news-data.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Fichier de données introuvable : {DATA_PATH.resolve()}")

expected_columns = ["category", "filename", "title", "content"]
actual_columns = pd.read_csv(DATA_PATH, sep="\t", nrows=0).columns.tolist()
print(f"\nFichier de données : {DATA_PATH.resolve()}")
print(f"Colonnes détectées : {actual_columns}")
missing_columns = [column for column in expected_columns if column not in actual_columns]
if missing_columns:
    raise ValueError(f"Colonnes attendues manquantes : {missing_columns}")
print("Vérification du fichier réussie : toutes les colonnes attendues sont présentes.")

Version de Python : 3.13.14
Attention : les consignes recommandent Python 3.13.5 ou une version antérieure pour Keras/TensorFlow.
pandas : 3.0.6
numpy : 2.5.3
scikit-learn : 1.9.1
nltk : 3.10.3
tensorflow : 2.21.0
torch : 2.14.1
gensim : 4.4.0

Fichier de données : C:\Users\boniv\Downloads\Github Project\ML-Trainning-and-evaluation\Apply SVM to predict BBC news category\bbc-news-data.csv
Colonnes détectées : ['category', 'filename', 'title', 'content']
Vérification du fichier réussie : toutes les colonnes attendues sont présentes.


## Étape 2 — Chargement et exploration du jeu de données

Nous chargeons le fichier tabulé dans un DataFrame, puis observons sa taille, les premières lignes, la répartition des catégories, les valeurs manquantes et les lignes dupliquées. La colonne `filename` sert uniquement d’identifiant et ne sera pas utilisée comme texte d’entrée du modèle.

**Objectif :** comprendre la structure et la qualité des données afin de savoir si un nettoyage sera nécessaire avant l’entraînement.

In [6]:
df = pd.read_csv(DATA_PATH, sep="\t")

print(f"Dimensions du jeu de données : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nTypes des colonnes :\n{df.dtypes}")
print("\nValeurs manquantes par colonne :")
display(df.isna().sum().to_frame(name="valeurs_manquantes"))

print(f"\nNombre de lignes dupliquées : {df.duplicated().sum()}")
print(f"\nNombre de catégories : {df['category'].nunique()}")
print("Répartition des catégories :")
display(df["category"].value_counts().rename_axis("category").to_frame(name="nombre_articles"))

print("\nAperçu de quelques articles :")
display(df[["category", "filename", "title", "content"]].head())

Dimensions du jeu de données : 2225 lignes × 4 colonnes

Types des colonnes :
category    str
filename    str
title       str
content     str
dtype: object

Valeurs manquantes par colonne :


,valeurs_manquantes
category,0
filename,0
title,0
content,0



Nombre de lignes dupliquées : 0

Nombre de catégories : 5
Répartition des catégories :


,nombre_articles
category,
sport,511
business,510
politics,417
tech,401
entertainment,386



Aperçu de quelques articles :


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


## Étape 3 — Prétraitement du texte avec NLTK

Nous réunissons le titre et le contenu dans `raw_text`, puis mettons le texte en minuscules, le tokenisons, conservons uniquement les mots composés de lettres, retirons les mots vides anglais et lemmatisons les tokens. Nous gardons les tokens nettoyés dans `tokens` et leur version réunie dans `processed_text` pour les vectoriseurs classiques.

**Objectif :** normaliser les articles pour réduire le bruit textuel et préparer des représentations adaptées aux différents modèles.

In [7]:
nltk_resources = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
    "stopwords": "corpora/stopwords",
    "wordnet": "corpora/wordnet",
    "omw-1.4": "corpora/omw-1.4",
}
for resource_name, resource_path in nltk_resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        if not nltk.download(resource_name, quiet=True):
            raise RuntimeError(f"Impossible de télécharger la ressource NLTK : {resource_name}")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    alphabetic_tokens = [token for token in tokens if token.isalpha()]
    return [
        lemmatizer.lemmatize(token)
        for token in alphabetic_tokens
        if token not in stop_words
    ]

df["raw_text"] = (
    df["title"].fillna("").astype(str).str.strip()
    + " "
    + df["content"].fillna("").astype(str).str.strip()
).str.strip()
df["tokens"] = df["raw_text"].apply(preprocess_text)
df["processed_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

display(df[["category", "raw_text", "processed_text"]].head(3))
print(f"Articles prétraités : {len(df)}")
print(f"Articles sans token après nettoyage : {(df['tokens'].str.len() == 0).sum()}")

,category,raw_text,processed_text
0,business,Ad sales boost Time Warner profit Quarterly pr...,ad sale boost time warner profit quarterly pro...
1,business,Dollar gains on Greenspan speech The dollar ha...,dollar gain greenspan speech dollar hit highes...
2,business,Yukos unit buyer faces loan claim The owners o...,yukos unit buyer face loan claim owner embattl...


Articles prétraités : 2225
Articles sans token après nettoyage : 0


## Étape 4 — Séparation des données en entraînement et test

Nous réservons 20 % des articles pour le test et utilisons une séparation stratifiée, afin de conserver la proportion de chaque catégorie dans les deux ensembles. La graine aléatoire fixe rend la séparation reproductible. Les mêmes indices sont appliqués au texte nettoyé, aux tokens et aux étiquettes.

**Objectif :** préparer une évaluation équitable sur des articles qui ne seront pas utilisés pendant l’entraînement. Les vectoriseurs devront ensuite être ajustés uniquement sur le texte d’entraînement.

In [8]:
SEED = 42
TEST_SIZE = 0.20
article_indices = np.arange(len(df))

train_indices, test_indices = train_test_split(
    article_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["category"],
)

X_train_text = df["processed_text"].iloc[train_indices].reset_index(drop=True)
X_test_text = df["processed_text"].iloc[test_indices].reset_index(drop=True)
X_train_tokens = df["tokens"].iloc[train_indices].tolist()
X_test_tokens = df["tokens"].iloc[test_indices].tolist()
y_train = df["category"].iloc[train_indices].reset_index(drop=True)
y_test = df["category"].iloc[test_indices].reset_index(drop=True)

if set(train_indices).intersection(test_indices):
    raise RuntimeError("Les ensembles d'entraînement et de test se chevauchent.")
if len(train_indices) + len(test_indices) != len(df):
    raise RuntimeError("La séparation n'a pas conservé tous les articles.")

print(f"Articles d'entraînement : {len(X_train_text)} ({len(X_train_text) / len(df):.1%})")
print(f"Articles de test : {len(X_test_text)} ({len(X_test_text) / len(df):.1%})")
print("\nRépartition des catégories (pourcentage dans chaque ensemble) :")
class_distribution = pd.concat(
    [
        y_train.value_counts(normalize=True).mul(100).rename("train (%)"),
        y_test.value_counts(normalize=True).mul(100).rename("test (%)"),
    ],
    axis=1,
).sort_index()
display(class_distribution.round(2))

Articles d'entraînement : 1780 (80.0%)
Articles de test : 445 (20.0%)

Répartition des catégories (pourcentage dans chaque ensemble) :


,train (%),test (%)
category,,
business,22.92,22.92
entertainment,17.36,17.30
politics,18.71,18.88
sport,22.98,22.92
tech,18.03,17.98
